In [ ]:
import marimo as mo

# QoS-HRGN: Heterogeneous Adaptive Graph Learning for BraTS 3D Segmentation

**Proposed pipeline**

`BraTS MRI → modality-specific 3D SLIC → heterogeneous supervoxel graph → HGT → adaptive multi-hop propagation → pathology-aware prediction → voxel reconstruction`

This notebook is the **research prototype**, not a claim that every component is already validated. The current implementation explicitly separates:
- **HGT** as the established heterogeneous graph-learning backbone.
- **Adaptive multi-hop propagation** as the proposed extension.
- **Pathology-aware prediction** using the BraTS NCR/NET, ED and ET labels and deterministic TC/WT construction.

BraTS 2020 provides T1, post-contrast T1 (T1Gd/T1ce), T2 and T2-FLAIR volumes and manual tumor annotations. The official labels are 0=background, 1=NCR/NET, 2=ED, 4=ET. TC and WT are evaluation regions derived from these labels. citeturn0search0turn0search1

## 1. Install dependencies

The notebook uses PyTorch Geometric's `HGTConv`, which accepts heterogeneous node/edge metadata and returns one embedding per node type. citeturn0search3

In [ ]:
# packages added via marimo's package management: torch-geometric nibabel scikit-image matplotlib kagglehub !pip install -q torch-geometric nibabel scikit-image matplotlib kagglehub

import os, glob, re, random, time, math
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

from skimage.segmentation import slic
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv
from torch_geometric.loader import DataLoader

print("PyTorch:", torch.__version__)
try:
    import torch_geometric
    print("PyG:", torch_geometric.__version__)
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# GPU performance settings.
USE_AMP = device.type == "cuda"
if USE_AMP:
    # Safe for modern NVIDIA GPUs; improves matrix-multiply throughput.
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"[Device] Using: {device.type.upper()}")
if USE_AMP:
    print("[Device] GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
    print(f"[Device] AMP enabled: {USE_AMP}")
else:
    pass

## 2. Download / discover BraTS data

The default source is the same BraTS 2020 NIfTI mirror used by the previous notebook. Set `MAX_CASES` to a small number while debugging, then increase it for the real experiment.

In [ ]:
import kagglehub
print("[1/8] Discovering BraTS cases...")
DATASET_SLUG = 'awsaf49/brats20-dataset-training-validation'
dataset_root = kagglehub.dataset_download(DATASET_SLUG)

def modality_kind(path):
    name = os.path.basename(path).lower()
    if re.search('(^|[_-])t1ce([_.-]|$)', name): return 't1ce'
    if re.search('(^|[_-])flair([_.-]|$)', name): return 'flair'
    if re.search('(^|[_-])seg([_.-]|$)', name): return 'seg'
    if re.search('(^|[_-])t2([_.-]|$)', name): return 't2'
    if re.search('(^|[_-])t1([_.-]|$)', name): return 't1'
    return None

all_nii = glob.glob(os.path.join(dataset_root, '**', '*.nii*'), recursive=True)
case_map = {}
for p in all_nii:
    k = modality_kind(p)
    if k: case_map.setdefault(os.path.dirname(p), {})[k] = p

MAX_CASES = 10
valid_cases_all = sorted([(os.path.basename(d), m) for d, m in case_map.items()
                          if all((k in m for k in ['t1ce', 'flair', 'seg']))])
selected_cases = valid_cases_all[:MAX_CASES]
assert len(selected_cases) == 10

print("==================================================")
print("DATASET INITIALIZATION")
print("==================================================")
print(f"Total valid cases discovered: {len(valid_cases_all)}")
print(f"Cases selected: {len(selected_cases)}\n")
print("Selected cases:")
for i, (cid, _) in enumerate(selected_cases):
    print(f"{i+1}. {cid}")
print("==================================================")


## 3. Utility functions

Each modality is normalized independently. SLIC is only a **regionalization step**: it groups neighboring voxels into supervoxels; it does not determine the tumor class.

In [ ]:
def load_nii(path):
    return nib.load(path).get_fdata().astype(np.float32)

def normalize_brain(vol):
    mask = vol > 0
    out = np.zeros_like(vol, dtype=np.float32)
    if mask.any():
        vals = vol[mask]
        out[mask] = (vals - vals.mean()) / (vals.std() + 1e-8)
    return out

RAW_TO_CLASS = {0: 0, 1: 1, 2: 2, 4: 3}
CLASS_NAMES = ["BG", "NCR/NET", "ED", "ET"]

def remap_labels(seg):
    out = np.zeros_like(seg, dtype=np.int64)
    out[seg == 1] = 1
    out[seg == 2] = 2
    out[seg == 4] = 3
    return out

def dice_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    return float((2 * inter + eps) / (pred.sum() + target.sum() + eps))

def segmentation_metrics(pred, target):
    # pred/target use 0,1,2,3 where 1=NCR/NET, 2=ED, 3=ET
    wt_p, wt_t = pred > 0, target > 0
    tc_p, tc_t = np.logical_or(pred == 1, pred == 3), np.logical_or(target == 1, target == 3)
    et_p, et_t = pred == 3, target == 3
    return {
        "Dice_WT": dice_binary(wt_p, wt_t),
        "Dice_TC": dice_binary(tc_p, tc_t),
        "Dice_ET": dice_binary(et_p, et_t),
    }

## 4. Build the proposed heterogeneous supervoxel graph

We create **two node types**:

- `t1ce`: supervoxels generated independently from T1ce.
- `flair`: supervoxels generated independently from FLAIR.

Relations:

- `t1ce --spatial--> t1ce`
- `flair --spatial--> flair`
- `t1ce --corresponds--> flair`
- `flair --corresponds--> t1ce`

The two modality-specific graphs preserve different regional boundaries, while correspondence edges let HGT exchange information between them.

For each node we store six features:

`[modality mean, modality std, normalized x, normalized y, normalized z, normalized volume]`.

For each edge we store a normalized spatial distance. The adaptive module later adds feature similarity and prediction uncertainty dynamically.

In [ ]:
def boundary_edges_from_segments(segments, brain_mask, id_to_idx):
    edges = set()
    for axis in range(3):
        a_sl = [slice(None)] * 3
        b_sl = [slice(None)] * 3
        a_sl[axis] = slice(0, -1)
        b_sl[axis] = slice(1, None)
        a = segments[tuple(a_sl)]
        b = segments[tuple(b_sl)]
        ma = brain_mask[tuple(a_sl)]
        mb = brain_mask[tuple(b_sl)]
        valid = ma & mb & (a != b)
        if valid.any():
            pairs = np.stack([a[valid], b[valid]], axis=1)
            for u, v in pairs:
                ui, vi = id_to_idx[int(u)], id_to_idx[int(v)]
                edges.add((ui, vi))
                edges.add((vi, ui))
    return list(edges)

def make_modality_nodes(volume, seg, mod_name, n_segments=200, compactness=0.1):
    print(f"  {mod_name} SLIC starting...")
    brain = volume != 0
    coords = np.argwhere(brain)
    lo = coords.min(axis=0)
    hi = coords.max(axis=0) + 1

    vol_c = volume[tuple(slice(lo[i], hi[i]) for i in range(3))]
    seg_c = seg[tuple(slice(lo[i], hi[i]) for i in range(3))]
    brain_c = brain[tuple(slice(lo[i], hi[i]) for i in range(3))]

    segments = slic(vol_c, n_segments=n_segments, compactness=compactness, max_num_iter=5, start_label=0, mask=brain_c, channel_axis=None)

    ids = np.unique(segments[brain_c])
    id_to_idx = {int(s): i for i, s in enumerate(ids)}
    print(f"  {mod_name} SLIC complete: {len(ids)} supervoxels")

    features, labels, centroids = [], [], []
    for sid in ids:
        mask = (segments == sid) & brain_c
        vals = vol_c[mask]
        xyz = np.argwhere(mask)
        centroid = xyz.mean(axis=0) / np.array(vol_c.shape[:3])
        raw = seg_c[mask]
        cls = np.zeros(4, dtype=np.int64)
        for rv in (0, 1, 2, 4): cls[RAW_TO_CLASS[rv]] = np.sum(raw == rv)
        features.append([float(vals.mean()), float(vals.std()), float(centroid[0]), float(centroid[1]), float(centroid[2]), float(mask.sum() / brain_c.sum())])
        labels.append(int(np.argmax(cls)))
        centroids.append(centroid)

    print(f"  Building {mod_name} spatial edges...")
    edges = boundary_edges_from_segments(segments, brain_c, id_to_idx)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    edge_dist = torch.tensor([np.linalg.norm(centroids[u] - centroids[v]) for u, v in edges], dtype=torch.float32).view(-1, 1) if edges else torch.empty((0, 1))
    if edge_dist.numel(): edge_dist = edge_dist / (edge_dist.mean() + 1e-8)
    print(f"  {mod_name} spatial edges complete: {len(edges)}")

    return {"x": torch.tensor(features, dtype=torch.float32), "y": torch.tensor(labels, dtype=torch.long), "edge_index": edge_index, "edge_dist": edge_dist, "centroids": np.asarray(centroids, dtype=np.float32), "segments": segments, "lo": lo, "hi": hi, "original_shape": volume.shape}

def build_hetero_case(case_files, n_segments=200, compactness=0.1):
    print("  Loading MRI modalities...")
    t1ce = normalize_brain(load_nii(case_files["t1ce"]))
    flair = normalize_brain(load_nii(case_files["flair"]))
    seg = remap_labels(load_nii(case_files["seg"]).astype(np.uint8))
    print("  MRI loading complete.\n")

    t1ce_g = make_modality_nodes(t1ce, seg, "T1ce", n_segments, compactness)
    print("")
    flair_g = make_modality_nodes(flair, seg, "FLAIR", n_segments, compactness)
    print("")

    print("  Building T1ce→FLAIR correspondence edges...")
    dmat = np.linalg.norm(t1ce_g["centroids"][:, None, :] - flair_g["centroids"][None, :, :], axis=-1)
    a_to_b = dmat.argmin(axis=1)
    cross_ab_list = [(i, int(j)) for i, j in enumerate(a_to_b)]
    print(f"  Correspondence edges complete: {len(cross_ab_list)}\n")
    
    print("  Building FLAIR→T1ce correspondence edges...")
    b_to_a = dmat.argmin(axis=0)
    cross_ba_list = [(int(i), j) for j, i in enumerate(b_to_a)]
    print(f"  Correspondence edges complete: {len(cross_ba_list)}\n")
    
    cross = list(set(cross_ab_list + cross_ba_list))

    cross_ab = torch.tensor(cross, dtype=torch.long).t().contiguous()
    cross_dist = torch.tensor([dmat[i, j] for i, j in cross], dtype=torch.float32).view(-1, 1)
    cross_dist = cross_dist / (cross_dist.mean() + 1e-8)

    data = HeteroData()
    data["t1ce"].x, data["t1ce"].y = t1ce_g["x"], t1ce_g["y"]
    data["flair"].x, data["flair"].y = flair_g["x"], flair_g["y"]
    data["t1ce", "spatial", "t1ce"].edge_index = t1ce_g["edge_index"]
    data["t1ce", "spatial", "t1ce"].edge_dist = t1ce_g["edge_dist"]
    data["flair", "spatial", "flair"].edge_index = flair_g["edge_index"]
    data["flair", "spatial", "flair"].edge_dist = flair_g["edge_dist"]
    data["t1ce", "corresponds", "flair"].edge_index = cross_ab
    data["t1ce", "corresponds", "flair"].edge_dist = cross_dist
    data["flair", "corresponds", "t1ce"].edge_index = cross_ab.flip(0)
    data["flair", "corresponds", "t1ce"].edge_dist = cross_dist.clone()

    print("  Graph construction complete.\n")
    print(f"  T1ce nodes: {data['t1ce'].num_nodes}")
    print(f"  FLAIR nodes: {data['flair'].num_nodes}")

    return data, {"t1ce": t1ce_g, "flair": flair_g, "seg": seg, "original_shape": seg.shape}


## 5. Inspect one heterogeneous graph

In [ ]:
sample_data, sample_meta = build_hetero_case(selected_cases[0][1], n_segments=200)

print(sample_data)
print("\nMetadata:", sample_data.metadata())
for nt in sample_data.node_types:
    print(nt, "nodes:", sample_data[nt].x.shape)
for et in sample_data.edge_types:
    print(et, "edges:", sample_data[et].edge_index.shape[1])

## 6. HGT + proposed adaptive multi-hop propagation

### HGT
`HGTConv` is the heterogeneous backbone. It receives separate node types and typed relations.

### Adaptive propagation
After HGT, we calculate an adaptive message weight for every edge from:

- source/destination embeddings
- normalized spatial distance
- feature similarity
- cross-modal relation indicator
- current prediction uncertainty

The adaptive layer is applied **twice**. Thus a node can receive information that has travelled beyond its immediate neighbors.

This is intentionally implemented as **learned graph propagation**, not literal packet routing or a shortest-path networking algorithm.

In [ ]:
class AdaptiveHeteroPropagation(nn.Module):
    def __init__(self, hidden_dim, edge_types):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.edge_types = edge_types

        self.msg = nn.ModuleDict()
        self.gate = nn.ModuleDict()

        for et in edge_types:
            key = "__".join(et)
            self.msg[key] = nn.Linear(hidden_dim, hidden_dim)
            # source + destination + 4 relation descriptors
            self.gate[key] = nn.Sequential(
                nn.Linear(hidden_dim * 2 + 4, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1)
            )

        self.norm = nn.ModuleDict({
            nt: nn.LayerNorm(hidden_dim) for nt in ["t1ce", "flair"]
        })

    def forward(self, x_dict, edge_index_dict, edge_dist_dict, logits_dict):
        out = {
            nt: torch.zeros_like(x_dict[nt]) for nt in x_dict
        }

        # Current uncertainty = normalized entropy of the 4-class prediction.
        unc = {}
        for nt, logits in logits_dict.items():
            p = torch.softmax(logits, dim=-1).clamp_min(1e-8)
            entropy = -(p * p.log()).sum(dim=-1) / math.log(4)
            unc[nt] = entropy

        for et in self.edge_types:
            src_type, rel, dst_type = et
            key = "__".join(et)
            ei = edge_index_dict[et]
            if ei.numel() == 0:
                continue

            src, dst = ei
            hs = x_dict[src_type][src]
            hd = x_dict[dst_type][dst]

            dist = edge_dist_dict[et].view(-1, 1).to(hs.device)
            sim = F.cosine_similarity(hs, hd, dim=-1).unsqueeze(1)
            cross = torch.full_like(dist, 1.0 if rel == "corresponds" else 0.0)
            u = ((unc[src_type][src] + unc[dst_type][dst]) / 2).unsqueeze(1)

            edge_features = torch.cat([hs, hd, dist, sim, cross, u], dim=1)
            alpha = torch.sigmoid(self.gate[key](edge_features))

            messages = self.msg[key](hs) * alpha

            # Aggregate source messages at destination nodes.
            out[dst_type].index_add_(0, dst, messages)

        result = {}
        for nt in x_dict:
            result[nt] = self.norm[nt](x_dict[nt] + F.relu(out[nt]))
        return result


class QoSHRGN(nn.Module):
    def __init__(self, in_dims, hidden_dim=64, heads=4, num_classes=4, hops=2):
        super().__init__()

        self.metadata = (
            ["t1ce", "flair"],
            [
                ("t1ce", "spatial", "t1ce"),
                ("flair", "spatial", "flair"),
                ("t1ce", "corresponds", "flair"),
                ("flair", "corresponds", "t1ce"),
            ]
        )

        self.input_proj = nn.ModuleDict({
            nt: nn.Linear(in_dims[nt], hidden_dim)
            for nt in in_dims
        })

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=self.metadata,
            heads=heads
        )
        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=self.metadata,
            heads=heads
        )

        self.pre_head = nn.ModuleDict({
            nt: nn.Linear(hidden_dim, num_classes) for nt in ["t1ce", "flair"]
        })

        self.adaptive = nn.ModuleList([
            AdaptiveHeteroPropagation(hidden_dim, self.metadata[1])
            for _ in range(hops)
        ])

        self.final_head = nn.ModuleDict({
            nt: nn.Linear(hidden_dim, num_classes) for nt in ["t1ce", "flair"]
        })

    def forward(self, data):
        x_dict = {
            nt: F.relu(self.input_proj[nt](data[nt].x))
            for nt in ["t1ce", "flair"]
        }

        edge_index_dict = {
            et: data[et].edge_index for et in self.metadata[1]
        }
        edge_dist_dict = {
            et: data[et].edge_dist for et in self.metadata[1]
        }

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x_dict = {
            nt: F.relu(x_dict[nt]) for nt in x_dict
        }
        x_dict = self.hgt2(x_dict, edge_index_dict)
        x_dict = {
            nt: F.relu(x_dict[nt]) for nt in x_dict
        }

        # Initial prediction supplies uncertainty to adaptive propagation.
        logits_dict = {
            nt: self.pre_head[nt](x_dict[nt]) for nt in x_dict
        }

        for layer in self.adaptive:
            x_dict = layer(
                x_dict,
                edge_index_dict,
                edge_dist_dict,
                logits_dict
            )
            logits_dict = {
                nt: self.pre_head[nt](x_dict[nt]) for nt in x_dict
            }

        final_logits = {
            nt: self.final_head[nt](x_dict[nt]) for nt in x_dict
        }
        return final_logits, x_dict

## 7. Pathology-aware loss

BraTS labels are **0, 1, 2, 4**, not four independent clinical classes. We train the node classifier on the three pathological subregions represented by the provided labels and derive:

- `ET = class 3`
- `TC = NCR/NET OR ET`
- `WT = NCR/NET OR ED OR ET`

This follows the BraTS task definition rather than inventing an independent NCR sink. citeturn0search0turn0search4

In [ ]:
def multiclass_dice_loss(probs, target, class_id, eps=1e-6):
    pred = probs[..., class_id]
    gt = (target == class_id).float()
    num = 2 * (pred * gt).sum() + eps
    den = pred.sum() + gt.sum() + eps
    return 1 - num / den

def hierarchy_loss(logits, target, class_weights=None):
    ce = F.cross_entropy(logits, target, weight=class_weights)

    probs = torch.softmax(logits, dim=-1)

    # Binary clinical regions.
    p_wt = 1 - probs[:, 0]
    p_tc = probs[:, 1] + probs[:, 3]
    p_et = probs[:, 3]

    gt_wt = (target > 0).float()
    gt_tc = ((target == 1) | (target == 3)).float()
    gt_et = (target == 3).float()

    def dice_loss_binary(p, g, eps=1e-6):
        return 1 - (2 * (p * g).sum() + eps) / (p.sum() + g.sum() + eps)

    d = (
        dice_loss_binary(p_wt, gt_wt)
        + dice_loss_binary(p_tc, gt_tc)
        + dice_loss_binary(p_et, gt_et)
    ) / 3.0

    return ce + 0.5 * d

## 8. Build the dataset graphs

The first run is intentionally limited to `MAX_CASES` for feasibility. Graphs are generated once and cached locally in Colab.

In [ ]:
def prepare_data(selected_cases, force_rebuild=False):
    print("[Cache] Checking 10-case graph cache...")
    CACHE = "qos_hrgn_10case_cache.pt"
    selected_ids = set(c[0] for c in selected_cases)

    if force_rebuild and os.path.exists(CACHE):
        os.remove(CACHE)
        print("[Cache] Stale cache deleted.")

    if os.path.exists(CACHE):
        try:
            graph_items = torch.load(CACHE, weights_only=False, map_location="cpu")
            cached_ids = set(meta['case_id'] for _, meta in graph_items)
            if cached_ids == selected_ids and len(graph_items) == 10:
                print("[Cache] Valid 10-case cache found.")
                print("[Cache] Loading...")
                print(f"[Cache] Loaded 10 graph cases.")
                return graph_items
        except Exception:
            pass
        print("[Cache] Cache missing or invalid.")

    print(f"[Cache] Rebuilding graphs for exactly 10 cases...")
    graph_items = []
    for i, (cid, files) in enumerate(selected_cases):
        print(f"\n[Graph {i+1}/10] Processing {cid}")
        d, meta = build_hetero_case(files, n_segments=200)
        meta['case_id'] = cid
        graph_items.append((d.cpu(), meta))
        print(f"[Graph {i+1}/10] Finished {cid}")

    torch.save(graph_items, CACHE)
    print("\n[Cache] Rebuild complete.")
    return graph_items

graph_items = prepare_data(selected_cases)
assert len(graph_items) == 10


## 9. Patient-level train/validation/test split

We split by **patient**, not by supervoxel, to avoid leaking regions from the same MRI volume into validation/test.

In [ ]:
random.seed(SEED)
random.shuffle(graph_items)

# Fixed 7/2/1 split for the 10 brains
train_items = graph_items[:7]
val_items = graph_items[7:9]
test_items = graph_items[9:]

assert len(train_items) == 7, f"Expected 7 train, got {len(train_items)}"
assert len(val_items)   == 2, f"Expected 2 val,   got {len(val_items)}"
assert len(test_items)  == 1, f"Expected 1 test,  got {len(test_items)}"
assert len(train_items) + len(val_items) + len(test_items) == 10
print(f"Split: Train={len(train_items)}, Val={len(val_items)}, Test={len(test_items)}")

counts = torch.zeros(4)
for d, _ in train_items:
    for nt in ['t1ce', 'flair']:
        counts += torch.bincount(d[nt].y, minlength=4).float()
class_weights = counts.sum() / (4 * counts.clamp_min(1))
class_weights = class_weights / class_weights.mean()
print('Class weights:', class_weights.tolist())


## 10. Train QoS-HRGN

This is the first actual end-to-end implementation of the proposed model:

`HGT → uncertainty-aware adaptive propagation × 2 → pathology prediction`.

No dummy patient-classification target is used.

In [ ]:
assert len(train_items) == 7
assert len(val_items) == 2
assert len(test_items) == 1

BATCH_SIZE = 1
model = QoSHRGN({'t1ce': 6, 'flair': 6}, hidden_dim=64, heads=4, num_classes=4, hops=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
class_weights = class_weights.to(device)

best_val_dice = -1.0
best_state = None
history = []

def run_sanity_checks(model, items):
    print("[Sanity] Running sanity checks...")
    assert len(items) == 10
    assert len(train_items) == 7 and len(val_items) == 2 and len(test_items) == 1

    data, meta = items[0]
    for nt in ['t1ce', 'flair']:
        assert nt in data.node_types
        assert data[nt].x.shape[1] == 6

    model.eval()
    with torch.no_grad():
        model(data.to(device))

    print("[Sanity] ALL CHECKS PASSED")

run_sanity_checks(model, graph_items)


## 11. Reconstruct node predictions into the original 3D voxel space

Each modality has its own SLIC partition. We therefore predict each modality's supervoxels and project those predictions back to voxels. Where both modality graphs provide a prediction, their class probabilities are averaged.

This gives an actual **3D segmentation volume**, not merely a graph-level classification.

In [ ]:
def reconstruct_case(model, data, meta):
    model.eval()
    assert np.array_equal(meta['t1ce']['lo'], meta['flair']['lo']), "Spatial alignment mismatch (lo)"
    assert np.array_equal(meta['t1ce']['hi'], meta['flair']['hi']), "Spatial alignment mismatch (hi)"

    with torch.no_grad():
        data_gpu = data.to(device)
        logits_dict, _ = model(data_gpu)
        p_t1ce_nodes = torch.softmax(logits_dict['t1ce'], dim=-1).cpu().numpy()
        p_flair_nodes = torch.softmax(logits_dict['flair'], dim=-1).cpu().numpy()

    def map_to_voxels(mod_meta, node_probs, mod_name):
        seg_map = np.asarray(mod_meta['segments'])
        prob_vol = np.zeros((*seg_map.shape, 4), dtype=np.float32)
        # Only iterate over brain segment IDs (SLIC sets masked-out voxels to -1)
        u_ids = np.unique(seg_map[seg_map >= 0])
        for idx, sid in enumerate(u_ids):
            prob_vol[seg_map == sid] = node_probs[idx]
        return prob_vol

    v_t1ce = map_to_voxels(meta['t1ce'], p_t1ce_nodes, "T1ce")
    v_flair = map_to_voxels(meta['flair'], p_flair_nodes, "FLAIR")
    combined_prob = 0.5 * (v_t1ce + v_flair)
    pred_seg = np.argmax(combined_prob, axis=-1)

    full_pred = np.zeros(meta['original_shape'], dtype=np.int64)
    lo, hi = meta['t1ce']['lo'], meta['t1ce']['hi']
    full_pred[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] = pred_seg

    return full_pred, combined_prob

## 12. Validation / test evaluation

BraTS evaluates **ET, TC and WT**, so these are the metrics we report rather than a meaningless patient-level accuracy.

In [ ]:
def evaluate_items(items, name="Test"):
    rows = []
    for idx, (data, meta) in enumerate(items):
        cid = meta.get('case_id', f'Case_{idx}')
        if name == "Test":
            print(f"[{name} {idx+1}/{len(items)}] {cid}")
            print(f"[{name}] Running inference...")
            print(f"[{name}] Reconstructing voxel prediction...")
        pred, _ = reconstruct_case(model, data, meta)
        if name == "Test":
            print(f"[{name}] Calculating WT/TC/ET Dice...")
        m = segmentation_metrics(pred, meta["seg"])
        rows.append(m)
        
        if name == "Test":
            m_mean = (m['Dice_WT'] + m['Dice_TC'] + m['Dice_ET']) / 3.0
            print(f"[{name}] WT={m['Dice_WT']:.4f}")
            print(f"[{name}] TC={m['Dice_TC']:.4f}")
            print(f"[{name}] ET={m['Dice_ET']:.4f}")
            print(f"[{name}] Mean={m_mean:.4f}")

    avg = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
    return avg


In [ ]:
print("\n==================================================")
print("TRAINING START")
print("==================================================")

def train_step(epochs=30):
    global best_val_dice, best_state
    loader = DataLoader([d for d, _ in train_items], batch_size=BATCH_SIZE, shuffle=True)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    for epoch in range(1, epochs + 1):
        print(f"[Train] Epoch {epoch}/{epochs} starting...")
        model.train()
        epoch_loss = 0
        for b_idx, data in enumerate(loader):
            data = data.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits, _ = model(data)
                loss = (hierarchy_loss(logits['t1ce'], data['t1ce'].y, class_weights) +
                        hierarchy_loss(logits['flair'], data['flair'].y, class_weights)) / 2
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            print(f"[Train] Epoch {epoch} | Batch {b_idx+1}/{len(loader)} | Loss={loss.item():.4f}")

        avg_l = epoch_loss / len(loader)
        history.append(avg_l)
        print(f"[Train] Epoch {epoch}/{epochs} complete | Loss={avg_l:.4f}")

        if epoch % 5 == 0 or epoch == 1:
            print(f"[Val] Epoch {epoch} starting...")
            v = evaluate_items(val_items, name="Val")
            m_dice = (v['Dice_WT'] + v['Dice_TC'] + v['Dice_ET']) / 3.0
            print(f"[Val] WT={v['Dice_WT']:.4f}")
            print(f"[Val] TC={v['Dice_TC']:.4f}")
            print(f"[Val] ET={v['Dice_ET']:.4f}")
            print(f"[Val] Mean={m_dice:.4f}")
            if m_dice > best_val_dice:
                best_val_dice = m_dice
                best_state = {k: val.cpu().clone() for k, val in model.state_dict().items()}
                print(f"[Checkpoint] New best model saved.")

train_step(30)
if best_state:
    model.load_state_dict(best_state)
    print("\n==================================================")
    print("BEST MODEL RESTORED FOR TESTING")
    print("==================================================")


## 13. Test evaluation and final summary

In [ ]:
print("==================================================")
print("TESTING START")
print("==================================================")

final_test_metrics = evaluate_items(test_items, "Test")
final_test_mean = (final_test_metrics['Dice_WT'] + final_test_metrics['Dice_TC'] + final_test_metrics['Dice_ET']) / 3.0

print("\n==================================================")
print("10-BRAIN EXPERIMENT COMPLETE")
print("==================================================\n")
print("Total brains: 10")
print(f"Train: {len(train_items)}")
print(f"Validation: {len(val_items)}")
print(f"Test: {len(test_items)}\n")
print("Best validation:")
print(f"Mean = {best_val_dice:.4f}\n")
print("Test:")
print(f"WT = {final_test_metrics['Dice_WT']:.4f}")
print(f"TC = {final_test_metrics['Dice_TC']:.4f}")
print(f"ET = {final_test_metrics['Dice_ET']:.4f}")
print(f"Mean = {final_test_mean:.4f}\n")
print("Status: COMPLETE")
print("==================================================")


## 14. Visualize one segmentation result

In [ ]:
def show_prediction(item):
    data, meta = item
    pred, _ = reconstruct_case(model, data, meta)
    target = meta["seg"]

    z = target.shape[2] // 2
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    ax[0].imshow(target[:, :, z].T, cmap="viridis", origin="lower", vmin=0, vmax=3)
    ax[0].set_title("Ground truth")

    ax[1].imshow(pred[:, :, z].T, cmap="viridis", origin="lower", vmin=0, vmax=3)
    ax[1].set_title("QoS-HRGN prediction")

    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()

show_prediction(test_items[0])


## 15. Save the trained model

The checkpoint contains the proposed architecture and learned parameters. The notebook can later be extended to run on the full BraTS training set.

In [ ]:
selected_case_ids = [m['case_id'] for _, m in graph_items]
checkpoint = {
    'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    'class_weights': class_weights.detach().cpu(),
    'selected_case_ids': selected_case_ids,
    'best_val_dice': best_val_dice,
    'config': {'hidden_dim': 64, 'heads': 4, 'hops': 2, 'num_classes': 4, 'n_segments': 200, 'batch_size': BATCH_SIZE}
}
torch.save(checkpoint, 'qos_hrgn_10case_final.pt')
print(f"Saved final checkpoint with {len(selected_case_ids)} cases to qos_hrgn_10case_final.pt")


## 16. What is actually implemented

### Existing backbone
- 3D SLIC supervoxels.
- Heterogeneous `HeteroData` graph.
- `T1ce` and `FLAIR` node types.
- Spatial and cross-modal relations.
- PyG `HGTConv`.

### Proposed component
- Adaptive propagation weights conditioned on node embeddings, spatial distance, feature similarity, cross-modal relation and prediction uncertainty.
- Two adaptive propagation stages, providing multi-hop information flow.
- Pathology-aware prediction and deterministic TC/WT construction.

### Important limitations
- This is a research prototype.
- The experiment should be run on a substantially larger patient cohort before making performance claims.
- Ablation experiments are required to demonstrate whether adaptive propagation actually improves over HGT alone.
- The novelty should be described as the **proposed adaptive propagation strategy on a heterogeneous multimodal supervoxel graph**, not as invention of HGT or SLIC.